In [2]:
import sys, subprocess
result = subprocess.run([sys.executable, '-m', 'pip', 'install', 'transformers', 'torch'], capture_output=True, text=True)
print(result.stdout[-2000:])
print(result.stderr[-2000:])

e-3.11-64\lib\site-packages (from typer->transformers) (0.0.5)
Using cached transformers-5.16.1-py3-none-any.whl (12.1 MB)
Using cached huggingface_hub-1.29.0-py3-none-any.whl (795 kB)
Using cached tokenizers-0.23.1-cp310-abi3-win_amd64.whl (2.8 MB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



In [3]:
from transformers import pipeline
classifier = pipeline("text-classification", model="facebook/roberta-hate-speech-dynabench-r4-target")
result = classifier("I hate everyone from that group, they should all leave")
print(result)

c:\Users\almas\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\almas\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\almas\.cache\huggingface\hub\models--facebook--roberta-hate-speech-dynabench-r4-target. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mod

[{'label': 'nothate', 'score': 0.9998563528060913}]


In [4]:
test_sentences = [
    "I hate everyone from that group, they should all leave",
    "Have a wonderful day, hope you're doing well!",
    "People like them don't deserve to exist",
    "I love spending time with my family",
]

for s in test_sentences:
    result = classifier(s)
    print(f"{s[:50]}... -> {result}")

I hate everyone from that group, they should all l... -> [{'label': 'nothate', 'score': 0.9998563528060913}]
Have a wonderful day, hope you're doing well!... -> [{'label': 'nothate', 'score': 0.999863862991333}]
People like them don't deserve to exist... -> [{'label': 'nothate', 'score': 0.999859094619751}]
I love spending time with my family... -> [{'label': 'nothate', 'score': 0.9998610019683838}]


In [5]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("facebook/roberta-hate-speech-dynabench-r4-target")
print(config.id2label)
print(config.label2id)

{0: 'nothate', 1: 'hate'}
None


In [6]:
test_sentences_v2 = [
    "I hate everyone from that group",  # vague — control, expect nothate or borderline
    "All immigrants are criminals and should be deported",  # explicit identity + hostility
    "Women shouldn't be allowed to work",  # explicit identity + hostility
    "Have a wonderful day, hope you're doing well!",  # clearly benign
    "I love spending time with my family",  # clearly benign
]

for s in test_sentences_v2:
    result = classifier(s)
    print(f"{s} -> {result}")

I hate everyone from that group -> [{'label': 'nothate', 'score': 0.9998677968978882}]
All immigrants are criminals and should be deported -> [{'label': 'hate', 'score': 0.999776303768158}]
Women shouldn't be allowed to work -> [{'label': 'hate', 'score': 0.999761164188385}]
Have a wonderful day, hope you're doing well! -> [{'label': 'nothate', 'score': 0.999863862991333}]
I love spending time with my family -> [{'label': 'nothate', 'score': 0.9998610019683838}]


In [7]:
from transformers import pipeline

toxicity_classifier = pipeline(
    "text-classification",
    model="unitary/toxic-bert",
    top_k=None  # returns all labels with scores, not just the top one
)

test_sentences_v3 = [
    "I hate everyone from that group, they should all leave",
    "You're a worthless piece of garbage",
    "All immigrants are criminals and should be deported",
    "Have a wonderful day, hope you're doing well!",
    "I love spending time with my family",
]

for s in test_sentences_v3:
    result = toxicity_classifier(s)
    print(f"{s}\n -> {result}\n")

c:\Users\almas\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\almas\.cache\huggingface\hub\models--unitary--toxic-bert. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6359.94it/s]


I hate everyone from that group, they should all leave
 -> [[{'label': 'toxic', 'score': 0.5922663807868958}, {'label': 'threat', 'score': 0.012232699431478977}, {'label': 'identity_hate', 'score': 0.011611642315983772}, {'label': 'insult', 'score': 0.011149127036333084}, {'label': 'obscene', 'score': 0.0030976259149610996}, {'label': 'severe_toxic', 'score': 0.0013175864005461335}]]

You're a worthless piece of garbage
 -> [[{'label': 'toxic', 'score': 0.9846166372299194}, {'label': 'insult', 'score': 0.9407455325126648}, {'label': 'obscene', 'score': 0.6997106075286865}, {'label': 'severe_toxic', 'score': 0.030201369896531105}, {'label': 'identity_hate', 'score': 0.011323070153594017}, {'label': 'threat', 'score': 0.0009648932027630508}]]

All immigrants are criminals and should be deported
 -> [[{'label': 'toxic', 'score': 0.8126072287559509}, {'label': 'identity_hate', 'score': 0.4722329378128052}, {'label': 'threat', 'score': 0.10628031939268112}, {'label': 'insult', 'score': 0.06

In [8]:
# Using your combined_tweets dataframe from earlier (has 'label' and 'text' columns)
bot_tweets = combined_tweets[combined_tweets['label'] == 1].copy()
print(bot_tweets.shape)

NameError: name 'combined_tweets' is not defined

In [9]:
import pandas as pd

base_path = r'C:\Users\almas\Downloads\cresci-2017.csv\datasets_full.csv\unzipped'

bot_folders = [
    'traditional_spambots_1',
    'social_spambots_1',
    'social_spambots_2',
    'social_spambots_3',
    'fake_followers',
]

bot_tweet_frames = []
for folder in bot_folders:
    path = f'{base_path}\\{folder}.csv\\{folder}.csv\\tweets.csv'
    df = pd.read_csv(path, encoding='latin1', low_memory=False)
    df['account_type'] = folder
    bot_tweet_frames.append(df)
    print(f"{folder}: {df.shape}")

bot_tweets = pd.concat(bot_tweet_frames, ignore_index=True)
print("Total bot tweets:", bot_tweets.shape)

traditional_spambots_1: (145094, 26)
social_spambots_1: (1610034, 26)
social_spambots_2: (428542, 26)
social_spambots_3: (1418557, 26)
fake_followers: (196027, 24)
Total bot tweets: (3798254, 26)


In [10]:
import numpy as np

np.random.seed(42)

# Cap at 30 tweets per account, randomly sampled
sampled_tweets = bot_tweets.groupby('user_id', group_keys=False).apply(
    lambda x: x.sample(min(len(x), 30), random_state=42)
)

print(sampled_tweets.shape)
print(sampled_tweets['account_type'].value_counts())

(232372, 25)
account_type
social_spambots_2         103710
fake_followers             69294
social_spambots_1          29730
traditional_spambots_1     15718
social_spambots_3          13920
Name: count, dtype: int64


In [11]:
# Sample a fixed number of tweets per account_type, preserving type diversity
sample_per_type = 500  # 500 tweets x 5 types = 2500 total

sampled_tweets = (
    bot_tweets.groupby('account_type', group_keys=False)
    .apply(lambda x: x.sample(min(len(x), sample_per_type), random_state=42))
)

print(sampled_tweets.shape)
print(sampled_tweets['account_type'].value_counts())

(2500, 25)


KeyError: 'account_type'

In [12]:
print(sampled_tweets.columns.tolist())

['id', 'text', 'source', 'user_id', 'truncated', 'in_reply_to_status_id', 'in_reply_to_user_id', 'in_reply_to_screen_name', 'retweeted_status_id', 'geo', 'place', 'contributors', 'retweet_count', 'reply_count', 'favorite_count', 'favorited', 'retweeted', 'possibly_sensitive', 'num_hashtags', 'num_urls', 'num_mentions', 'created_at', 'timestamp', 'crawled_at', 'updated']


In [13]:
sampled_tweets = sampled_tweets.reset_index(drop=True)
print(sampled_tweets.columns.tolist())

['id', 'text', 'source', 'user_id', 'truncated', 'in_reply_to_status_id', 'in_reply_to_user_id', 'in_reply_to_screen_name', 'retweeted_status_id', 'geo', 'place', 'contributors', 'retweet_count', 'reply_count', 'favorite_count', 'favorited', 'retweeted', 'possibly_sensitive', 'num_hashtags', 'num_urls', 'num_mentions', 'created_at', 'timestamp', 'crawled_at', 'updated']


In [14]:
sample_per_type = 500

sampled_frames = []
for acc_type in bot_tweets['account_type'].unique():
    subset = bot_tweets[bot_tweets['account_type'] == acc_type]
    sampled = subset.sample(min(len(subset), sample_per_type), random_state=42)
    sampled_frames.append(sampled)

sampled_tweets = pd.concat(sampled_frames, ignore_index=True)

print(sampled_tweets.shape)
print(sampled_tweets['account_type'].value_counts())

(2500, 26)
account_type
traditional_spambots_1    500
social_spambots_1         500
social_spambots_2         500
social_spambots_3         500
fake_followers            500
Name: count, dtype: int64


In [15]:
from tqdm import tqdm
tqdm.pandas()

def get_hate_label(text):
    try:
        result = classifier(str(text)[:512])[0]  # truncate long tweets to avoid token limit issues
        return result['label'], result['score']
    except Exception:
        return None, None

def get_toxicity_scores(text):
    try:
        result = toxicity_classifier(str(text)[:512])[0]
        return {r['label']: r['score'] for r in result}
    except Exception:
        return {}

# Run hate-speech classifier
hate_results = sampled_tweets['text'].progress_apply(get_hate_label)
sampled_tweets['hate_label'] = hate_results.apply(lambda x: x[0])
sampled_tweets['hate_score'] = hate_results.apply(lambda x: x[1])

# Run toxicity classifier
tox_results = sampled_tweets['text'].progress_apply(get_toxicity_scores)
sampled_tweets['toxic_score'] = tox_results.apply(lambda x: x.get('toxic', None))
sampled_tweets['insult_score'] = tox_results.apply(lambda x: x.get('insult', None))
sampled_tweets['identity_hate_score'] = tox_results.apply(lambda x: x.get('identity_hate', None))
sampled_tweets['threat_score'] = tox_results.apply(lambda x: x.get('threat', None))

print(sampled_tweets[['text', 'hate_label', 'hate_score', 'toxic_score']].head(10))

100%|██████████| 2500/2500 [02:10<00:00, 19.14it/s]

                                                text hate_label  hate_score  \
0                                 feliz setembro! qq    nothate    0.997153   
1  Help this dogs friends with every book sold. h...    nothate    0.999301   
2  Non-Agricultural Products approved via HSE Che...    nothate    0.999668   
3  Mais seguidores e amizades no twitter? usem o ...    nothate    0.984243   
4  Asset allocation strategies for angel investor...    nothate    0.999348   
5  C\xe1o v\xe0 s\u1ebfu: \nC\xe1o m\u1eddi S\u1e...    nothate    0.986843   
6  "Virally Add Thousands Of New Followers To You...    nothate    0.994501   
7  Classifica seus amigos como um bando de? \u201...    nothate    0.997732   
8                       meee iindicaa aii pessoooaas       hate    0.888001   
9  Join me in swom and rock the world  http://win...    nothate    0.999376   

   toxic_score  
0     0.014132  
1     0.008236  
2     0.000613  
3     0.004019  
4     0.000592  
5     0.864184  
6     0.001

In [16]:
summary = sampled_tweets.groupby('account_type').agg(
    hate_rate=('hate_label', lambda x: (x == 'hate').mean()),
    avg_toxic_score=('toxic_score', 'mean'),
    high_toxic_rate=('toxic_score', lambda x: (x > 0.5).mean()),
    avg_insult_score=('insult_score', 'mean'),
    avg_identity_hate_score=('identity_hate_score', 'mean'),
    avg_threat_score=('threat_score', 'mean'),
).round(4)

print(summary)

                        hate_rate  avg_toxic_score  high_toxic_rate  \
account_type                                                          
fake_followers              0.044           0.0840            0.066   
social_spambots_1           0.010           0.0926            0.026   
social_spambots_2           0.042           0.0942            0.082   
social_spambots_3           0.022           0.0242            0.014   
traditional_spambots_1      0.004           0.0601            0.048   

                        avg_insult_score  avg_identity_hate_score  \
account_type                                                        
fake_followers                    0.0201                   0.0067   
social_spambots_1                 0.0097                   0.0031   
social_spambots_2                 0.0218                   0.0052   
social_spambots_3                 0.0040                   0.0010   
traditional_spambots_1            0.0062                   0.0008   

                  

In [17]:
ff_hate = sampled_tweets[(sampled_tweets['account_type'] == 'fake_followers') & (sampled_tweets['hate_label'] == 'hate')]
print(ff_hate[['text', 'hate_score']])

                                                   text  hate_score
2023  @hallsy04 hey taylor get well soon sucks it ha...    0.784103
2052  @st3phylee well, you are worse... you speed an...    0.709787
2116                               Slppppp z zzz zZzz z    0.999404
2149  NiallGetWellSoon#NiallGetWellSoon is the fucki...    0.908908
2167  YouHaveNoFriendsBecause#YouHaveNoFriendsBecaus...    0.999002
2226                                 Booaaa tardeee *-*    0.973250
2255  ILied#ILied when  I said you were badd it was ...    0.999431
2258  On enchainee au max... Linandes & la mosquÃ©e ...    0.701302
2265      Boo Hoo..Sad Story..Black American Dad Story.    0.619566
2305                             C!nt@ tak butuh bualan    0.997803
2313                   RT @Jay6373: Where my niggas at     0.983352
2346  Sir Edward Appleton~ I dont mind what language...    0.994763
2348  RT @Bugsey67: I think @jvejercito shd trend al...    0.740985
2367             Helloooo retweet if you're a be

In [18]:
genuine_path = f'{base_path}\\genuine_accounts.csv\\genuine_accounts.csv\\tweets.csv'
genuine_tweets = pd.read_csv(genuine_path, encoding='latin1', low_memory=False)
print(genuine_tweets.shape)

sample_genuine = genuine_tweets.sample(min(len(genuine_tweets), 2500), random_state=42)
print(sample_genuine.shape)

(2839362, 25)
(2500, 25)


In [19]:
sample_genuine = sample_genuine.copy()
sample_genuine['account_type'] = 'genuine_accounts'

hate_results_g = sample_genuine['text'].progress_apply(get_hate_label)
sample_genuine['hate_label'] = hate_results_g.apply(lambda x: x[0])
sample_genuine['hate_score'] = hate_results_g.apply(lambda x: x[1])

tox_results_g = sample_genuine['text'].progress_apply(get_toxicity_scores)
sample_genuine['toxic_score'] = tox_results_g.apply(lambda x: x.get('toxic', None))
sample_genuine['insult_score'] = tox_results_g.apply(lambda x: x.get('insult', None))
sample_genuine['identity_hate_score'] = tox_results_g.apply(lambda x: x.get('identity_hate', None))
sample_genuine['threat_score'] = tox_results_g.apply(lambda x: x.get('threat', None))

print(sample_genuine[['text', 'hate_label', 'hate_score', 'toxic_score']].head(10))

100%|██████████| 2500/2500 [01:57<00:00, 21.27it/s]

                                                      text hate_label  \
1317336  There's an electricity in the air at @GTBicycl...    nothate   
2309612                      @darkpoole @MGeschwind Agree.    nothate   
1898199  RT @bryanchaney: .@Twilio is hiring a #Recruit...    nothate   
1694427  Cuando se espera que alguien llegue , pero no ...    nothate   
2628190  I was looking forward to seeing you. My rival ...    nothate   
2353693                   And the inevitable pinhole burns    nothate   
2334523                                  Town business lol    nothate   
936438   First stop, Alcoy. http://t.co/e72PqFtBE8 http...    nothate   
181929   RT @MyVaranasi: à¤à¤¯à¤¾à¤ªà¥à¤° : à¤à¤¤à¥...    nothate   
836131   Sorry to everyone on The Terrace who just saw ...    nothate   

         hate_score  toxic_score  
1317336    0.996651     0.001801  
2309612    0.987688     0.000874  
1898199    0.990508     0.000809  
1694427    0.998864     0.054791  
2628190    0.999859  

In [20]:
combined_all = pd.concat([sampled_tweets, sample_genuine], ignore_index=True)

final_summary = combined_all.groupby('account_type').agg(
    hate_rate=('hate_label', lambda x: (x == 'hate').mean()),
    avg_toxic_score=('toxic_score', 'mean'),
    high_toxic_rate=('toxic_score', lambda x: (x > 0.5).mean()),
).round(4)

print(final_summary)

                        hate_rate  avg_toxic_score  high_toxic_rate
account_type                                                       
fake_followers              0.044           0.0840           0.0660
genuine_accounts            0.058           0.0985           0.0836
social_spambots_1           0.010           0.0926           0.0260
social_spambots_2           0.042           0.0942           0.0820
social_spambots_3           0.022           0.0242           0.0140
traditional_spambots_1      0.004           0.0601           0.0480


In [21]:
genuine_hate = sample_genuine[sample_genuine['hate_label'] == 'hate']
print(genuine_hate[['text', 'hate_score']])

                                                      text  hate_score
573246   @BillisKing it is hell getting old huh?Sounds ...    0.996250
1305092         @MexWhoCan hahahaha YES! ugh I'm horrible     0.998590
2473575  â@girlposts: Me: Baby I cooked  Bae: What's ...    0.997349
2317957  @RaquelYdolis @aboubakar47 fatass vs boney chick     0.885679
1894521  RT @fucktyler: BITCHES IN THEIR STUPID FESTIVA...    0.996561
...                                                    ...         ...
1311006  This Grimm Halloween special kinda like those ...    0.906927
970623               ä»æ¥ã¯ãã¼ã¹ã2ã¯ã¬ãã¼ã     0.835054
2019184  RT @YoungBLKRepub: Wow an epic takedown of a f...    0.558160
2780497  YEA I DO MY THANG BITCH WASSUP, YOUNG BASEDGOD...    0.964646
56904    What? How? Wtf.... They don't deserve it..... ...    0.999592

[145 rows x 2 columns]
